In [7]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    
    for file in files[:20]:
        print(f"{indent}  {file}")

working/
  .virtual_documents/
    __notebook_source__.ipynb


In [7]:
comp_dir = "/kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge/public"

carb_dir = "/kaggle/input/datasets/zhihaow/carbonglobe"

In [8]:
import os

train_file = f"{carb_dir}/data_global/res_train4_test8.npz"

print("\nTraining file exists:", os.path.exists(train_file))


Training file exists: True


In [9]:
import numpy as np

train_data = np.load(train_file)

print("Files inside res_train4_test8.npz:")
print(train_data.files)

Files inside res_train4_test8.npz:
['x_train', 'y_train', 'x_test', 'y_test']
x_train: (3373, 40, 12, 136)
y_train: (15, 3373, 41, 12, 7)
x_test: (852, 40, 12, 136)
y_test: (15, 852, 41, 12, 7)


In [10]:
import numpy as np

stats_file = f"{carb_dir}/data_stats/data_stats.npz"
stats = np.load(stats_file)

x_mean = stats["x_mean"]
x_std = stats["x_std"]

y_mean = stats["y_mean"]
y_std = stats["y_std"]

print(x_mean.shape)
print(x_std.shape)
print(y_mean.shape)
print(y_std.shape)


(136,)
(136,)
(7,)
(7,)


In [34]:
xtrain = (train_data['x_train'] - x_mean) / (x_std + 1e-10)

ytrain = (train_data['y_train'] - y_mean) / (y_std + 1e-10)

In [35]:
xtrain = xtrain.mean(axis=2)
xtrain.shape

(3373, 40, 136)

In [36]:
ytrain = ytrain[:, :, :, -1, :]
print(ytrain.shape)

(15, 3373, 41, 7)


In [14]:
ages = np.array([
    1,
    10,
    20,
    30,
    50,
    70,
    100,
    150,
    200,
    250,
    300,
    350,
    400,
    450,
    500
], dtype=np.float32)

print("Number of ages:", len(ages))
print("Ages:", ages)

Number of ages: 15
Ages: [  1.  10.  20.  30.  50.  70. 100. 150. 200. 250. 300. 350. 400. 450.
 500.]


# TRANSFORMER MODELS

In [16]:
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [37]:
num_train_sites = 3035

train_sites = np.arange(
    0,
    num_train_sites
)

val_sites = np.arange(
    num_train_sites,
    3373
)

print(
    "Training sites:",
    len(train_sites)
)

print(
    "Validation sites:",
    len(val_sites)
)

Training sites: 3035
Validation sites: 338


In [38]:
class CarbonGlobeDataset(Dataset):

    def __init__(
        self,
        xtrain,
        ytrain,
        site_indices,
        ages
    ):

        self.xtrain = xtrain
        self.ytrain = ytrain
        self.site_indices = site_indices
        self.ages = ages

        self.samples = [
            (site_idx, age_idx)
            for site_idx in site_indices
            for age_idx in range(len(ages))
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        site_idx, age_idx = self.samples[idx]

        # ------------------------------------------------
        # Environmental input
        # ------------------------------------------------

        env = self.xtrain[site_idx]

        # Shape:
        # (40, 136)

        # ------------------------------------------------
        # Previous ecosystem state
        # ------------------------------------------------

        previous_state = self.ytrain[
            age_idx,
            site_idx,
            :-1
        ]

        # Shape:
        # (40, 7)

        # ------------------------------------------------
        # Target ecosystem state
        # ------------------------------------------------

        target = self.ytrain[
            age_idx,
            site_idx,
            1:
        ]

        # Shape:
        # (40, 7)

        # ------------------------------------------------
        # Seed age
        # ------------------------------------------------

        age_value = self.ages[
            age_idx
        ] / 500.0

        age = np.full(
            (40, 1),
            age_value,
            dtype=np.float32
        )

        return (
            torch.from_numpy(env).float(),
            torch.from_numpy(previous_state).float(),
            torch.from_numpy(target).float(),
            torch.from_numpy(age).float()
        )

In [39]:
train_dataset = CarbonGlobeDataset(
    xtrain=xtrain,
    ytrain=ytrain,
    site_indices=train_sites,
    ages=ages
)

val_dataset = CarbonGlobeDataset(
    xtrain=xtrain,
    ytrain=ytrain,
    site_indices=val_sites,
    ages=ages
)

print(
    "Training sequences:",
    len(train_dataset)
)

print(
    "Validation sequences:",
    len(val_dataset)
)

Training sequences: 45525
Validation sequences: 5070


In [40]:
batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

Training batches: 2846
Validation batches: 317


In [41]:
class CarbonGlobeTransformer(nn.Module):

    def __init__(
        self,
        env_dim=136,
        state_dim=7,
        age_dim=1,
        d_model=256,
        n_heads=8,
        n_layers=4,
        d_ff=1024,
        dropout=0.1,
        max_years=40,
        output_dim=7
    ):

        super().__init__()

        # Environmental embedding
        self.env_embedding = nn.Sequential(
            nn.Linear(env_dim, 192),
            nn.LayerNorm(192),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Previous state embedding
        self.state_embedding = nn.Sequential(
            nn.Linear(state_dim, 48),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Age embedding
        self.age_embedding = nn.Sequential(
            nn.Linear(age_dim, 16),
            nn.LayerNorm(16),
            nn.GELU()
        )

        # Combine everything
        self.input_projection = nn.Sequential(
            nn.Linear(192 + 48 + 16, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Year positional embedding
        self.year_embedding = nn.Embedding(
            max_years,
            d_model
        )

        # Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers,
            norm=nn.LayerNorm(d_model)
        )

        # Delta prediction head
        self.delta_head = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, output_dim)
        )

    def forward(
        self,
        env,
        previous_state,
        age
    ):

        B, T, _ = env.shape

        env_emb = self.env_embedding(env)

        state_emb = self.state_embedding(
            previous_state
        )

        age_emb = self.age_embedding(age)

        x = torch.cat(
            [
                env_emb,
                state_emb,
                age_emb
            ],
            dim=-1
        )

        x = self.input_projection(x)

        # Year embeddings
        years = torch.arange(
            T,
            device=x.device
        )

        x = (
            x
            + self.year_embedding(years)
            .unsqueeze(0)
        )

        # Causal mask
        causal_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        h = self.transformer(
            x,
            mask=causal_mask
        )

        # Predict change from previous state
        delta = self.delta_head(h)

        prediction = (
            previous_state + delta
        )

        return prediction

In [42]:
model = CarbonGlobeTransformer(
    env_dim=136,
    state_dim=7,
    age_dim=1,

    d_model=256,
    n_heads=8,
    n_layers=4,
    d_ff=1024,

    dropout=0.1,
    max_years=40,
    output_dim=7
).to(device)

print(model)

CarbonGlobeTransformer(
  (env_embedding): Sequential(
    (0): Linear(in_features=136, out_features=192, bias=True)
    (1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
  )
  (state_embedding): Sequential(
    (0): Linear(in_features=7, out_features=48, bias=True)
    (1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
  )
  (age_embedding): Sequential(
    (0): Linear(in_features=1, out_features=16, bias=True)
    (1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
  )
  (input_projection): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
  )
  (year_embedding): Embedding(40, 256)
  (transformer): TransformerEncoder(
    (layer

/tmp/ipykernel_58/2970213671.py:67: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


In [44]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

Trainable parameters: 3297383


In [45]:
def carbon_loss(
    prediction,
    target,
    previous_state
):

    # State error
    state_loss = nn.functional.smooth_l1_loss(
        prediction,
        target
    )

    # True change
    true_delta = (
        target - previous_state
    )

    # Predicted change
    pred_delta = (
        prediction - previous_state
    )

    # Delta error
    delta_loss = nn.functional.smooth_l1_loss(
        pred_delta,
        true_delta
    )

    # Give later years slightly more weight
    T = prediction.shape[1]

    weights = torch.linspace(
        1.0,
        2.0,
        T,
        device=prediction.device
    ).view(1, T, 1)

    weighted_state_loss = (
        torch.abs(
            prediction - target
        ) * weights
    ).mean()

    # Total
    loss = (
        state_loss
        + 0.5 * delta_loss
        + 0.2 * weighted_state_loss
    )

    return loss

In [46]:
num_epochs = 30

best_val_loss = float("inf")

print(
    "Epochs:",
    num_epochs
)

print(
    "Batch size:",
    batch_size
)

print(
    "Learning rate:",
    1e-4
)

Epochs: 30
Batch size: 16
Learning rate: 0.0001


In [47]:
for epoch in range(num_epochs):

    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    train_loss = 0.0

    for batch_idx, (
        X_batch,
        Y_previous_batch,
        Y_target_batch,
        age_batch
    ) in enumerate(train_loader):

        # ----------------------------------------------------
        # Move to GPU
        # ----------------------------------------------------

        X_batch = X_batch.to(
            device,
            non_blocking=True
        )

        Y_previous_batch = Y_previous_batch.to(
            device,
            non_blocking=True
        )

        Y_target_batch = Y_target_batch.to(
            device,
            non_blocking=True
        )

        age_batch = age_batch.to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Clear old gradients
        # ----------------------------------------------------

        optimizer.zero_grad()

        # ----------------------------------------------------
        # FORWARD
        # ----------------------------------------------------

        prediction = model(
            env=X_batch,
            previous_state=Y_previous_batch,
            age=age_batch
        )

        # ----------------------------------------------------
        # LOSS
        # ----------------------------------------------------

        loss = carbon_loss(
            prediction,
            Y_target_batch,
            Y_previous_batch
        )

        # ----------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------

        loss.backward()

        # ----------------------------------------------------
        # GRADIENT CLIPPING
        # ----------------------------------------------------

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # ----------------------------------------------------
        # UPDATE WEIGHTS
        # ----------------------------------------------------

        optimizer.step()

        train_loss += loss.item()

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (batch_idx + 1) % 100 == 0:

            print(
                f"Epoch {epoch+1}/{num_epochs} | "
                f"Batch {batch_idx+1}/{len(train_loader)} | "
                f"Loss {loss.item():.6f}"
            )

    train_loss /= len(train_loader)


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for (
            X_batch,
            Y_previous_batch,
            Y_target_batch,
            age_batch
        ) in val_loader:

            X_batch = X_batch.to(
                device,
                non_blocking=True
            )

            Y_previous_batch = Y_previous_batch.to(
                device,
                non_blocking=True
            )

            Y_target_batch = Y_target_batch.to(
                device,
                non_blocking=True
            )

            age_batch = age_batch.to(
                device,
                non_blocking=True
            )

            prediction = model(
                env=X_batch,
                previous_state=Y_previous_batch,
                age=age_batch
            )

            loss = carbon_loss(
                prediction,
                Y_target_batch,
                Y_previous_batch
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)


    # ========================================================
    # EPOCH SUMMARY
    # ========================================================

    print()
    print("=" * 60)
    print(
        f"Epoch {epoch+1}/{num_epochs}"
    )
    print(
        f"Train Loss: {train_loss:.6f}"
    )
    print(
        f"Val Loss:   {val_loss:.6f}"
    )
    print("=" * 60)


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "train_loss":
                    train_loss,
                "val_loss":
                    val_loss
            },
            "carbon_globe_transformer_best.pt"
        )

        print(
            "✓ Best model saved"
        )

    print()

Epoch 1/30 | Batch 100/2846 | Loss 0.039254
Epoch 1/30 | Batch 200/2846 | Loss 0.034046
Epoch 1/30 | Batch 300/2846 | Loss 0.039429
Epoch 1/30 | Batch 400/2846 | Loss 0.029226
Epoch 1/30 | Batch 500/2846 | Loss 0.053246
Epoch 1/30 | Batch 600/2846 | Loss 0.037915
Epoch 1/30 | Batch 700/2846 | Loss 0.010627
Epoch 1/30 | Batch 800/2846 | Loss 0.031721
Epoch 1/30 | Batch 900/2846 | Loss 0.010879
Epoch 1/30 | Batch 1000/2846 | Loss 0.014563
Epoch 1/30 | Batch 1100/2846 | Loss 0.059814
Epoch 1/30 | Batch 1200/2846 | Loss 0.022861
Epoch 1/30 | Batch 1300/2846 | Loss 0.050082
Epoch 1/30 | Batch 1400/2846 | Loss 0.029597
Epoch 1/30 | Batch 1500/2846 | Loss 0.018247
Epoch 1/30 | Batch 1600/2846 | Loss 0.022923
Epoch 1/30 | Batch 1700/2846 | Loss 0.048063
Epoch 1/30 | Batch 1800/2846 | Loss 0.033903
Epoch 1/30 | Batch 1900/2846 | Loss 0.026730
Epoch 1/30 | Batch 2000/2846 | Loss 0.022543
Epoch 1/30 | Batch 2100/2846 | Loss 0.033716
Epoch 1/30 | Batch 2200/2846 | Loss 0.034223
Epoch 1/30 | Batch 

In [48]:
checkpoint = torch.load(
    "carbon_globe_transformer_best.pt",
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print(
    "Best epoch:",
    checkpoint["epoch"]
)

print(
    "Best validation loss:",
    checkpoint["val_loss"]
)

Best epoch: 5
Best validation loss: 0.05726692123887238


In [50]:
test_xtrain = train_data["x_test"].mean(
    axis=2
).astype(np.float32)

test_xtrain = (
    test_xtrain - x_mean
) / (
    x_std + 1e-10
)

test_ytrain = train_data["y_test"][
    :, :, :, -1, :
].astype(np.float32)

test_ytrain = (
    test_ytrain - y_mean
) / (
    y_std + 1e-10
)

print(
    "Test X:",
    test_xtrain.shape
)

print(
    "Test Y:",
    test_ytrain.shape
)

Test X: (852, 40, 136)
Test Y: (15, 852, 41, 7)


In [55]:
test_ages = np.array([
    1, 10, 20, 30, 50,
    70, 100, 150, 200, 250,
    300, 350, 400, 450, 500
], dtype=np.float32)

test_ages = test_ages / 500.0

In [56]:
checkpoint = torch.load(
    "carbon_globe_transformer_best.pt",
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

CarbonGlobeTransformer(
  (env_embedding): Sequential(
    (0): Linear(in_features=136, out_features=192, bias=True)
    (1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
  )
  (state_embedding): Sequential(
    (0): Linear(in_features=7, out_features=48, bias=True)
    (1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
  )
  (age_embedding): Sequential(
    (0): Linear(in_features=1, out_features=16, bias=True)
    (1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
  )
  (input_projection): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
  )
  (year_embedding): Embedding(40, 256)
  (transformer): TransformerEncoder(
    (layer

In [58]:
predictions = []

model.eval()

with torch.no_grad():

    for age_idx in range(15):

        # Environmental input
        env = torch.from_numpy(
            test_xtrain
        ).float().to(device)

        # Actual year-0 December state
        previous_state = torch.from_numpy(
            test_ytrain[age_idx, :, 0, :]
        ).float().to(device)

        # Constant seed age for all 40 years
        age = torch.full(
            (852, 40, 1),
            float(test_ages[age_idx]),
            dtype=torch.float32,
            device=device
        )

        age_predictions = []

        for year in range(40):

            current_env = env[:, year:year+1, :]
            current_state = previous_state.unsqueeze(1)
            current_age = age[:, year:year+1, :]

            pred = model(
                env=current_env,
                previous_state=current_state,
                age=current_age
            )

            pred = pred[:, 0, :]

            age_predictions.append(
                pred.cpu().numpy()
            )

            # Autoregressive:
            # predicted state becomes next year's input
            previous_state = pred

        age_predictions = np.stack(
            age_predictions,
            axis=1
        )

        predictions.append(
            age_predictions
        )

predictions = np.stack(
    predictions,
    axis=0
)

print("Predictions:", predictions.shape)

Predictions: (15, 852, 40, 7)


In [59]:
print("Model dtype:", next(model.parameters()).dtype)
print("X dtype:", test_xtrain.dtype)
print("Y dtype:", test_ytrain.dtype)

Model dtype: torch.float32
X dtype: float64
Y dtype: float64


In [60]:
true_targets = test_ytrain[:, :, 1:, :]

print("True:", true_targets.shape)
print("Pred:", predictions.shape)

True: (15, 852, 40, 7)
Pred: (15, 852, 40, 7)


In [61]:
rmse = np.sqrt(
    np.mean(
        (predictions - true_targets) ** 2
    )
)

mae = np.mean(
    np.abs(predictions - true_targets)
)

print("Normalized RMSE:", rmse)
print("Normalized MAE :", mae)

Normalized RMSE: 0.4053768970361588
Normalized MAE : 0.18532579549572953


In [63]:
pred_physical = (
    predictions * (y_std + 1e-10)
) + y_mean

true_physical = (
    true_targets * (y_std + 1e-10)
) + y_mean

In [64]:
target_names = [
    "height",
    "agb",
    "soil",
    "lai",
    "gpp",
    "npp",
    "rh"
]

for i, name in enumerate(target_names):

    rmse = np.sqrt(
        np.mean(
            (
                pred_physical[:, :, :, i]
                -
                true_physical[:, :, :, i]
            ) ** 2
        )
    )

    mae = np.mean(
        np.abs(
            pred_physical[:, :, :, i]
            -
            true_physical[:, :, :, i]
        )
    )

    print(
        f"{name:6s} | "
        f"RMSE: {rmse:.4f} | "
        f"MAE: {mae:.4f}"
    )

height | RMSE: 2.9597 | MAE: 1.4524
agb    | RMSE: 1.2547 | MAE: 0.5905
soil   | RMSE: 1.0790 | MAE: 0.6218
lai    | RMSE: 0.8900 | MAE: 0.4223
gpp    | RMSE: 0.8198 | MAE: 0.3731
npp    | RMSE: 0.4018 | MAE: 0.1819
rh     | RMSE: 0.2803 | MAE: 0.1551
